In [ ]:
import os
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer
import json

LLM_MODEL_NAME="jaro-jm/llm:latest"
# LLM_MODEL_NAME="SpeakLeash/bielik-11b-v2.3-instruct:Q8_0"
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"

llm_text = ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=25000, num_predict=25000)

text = """Donald Trump od początku traktował konflikt w Ukrainie jako nieswoją wojnę. Nie miał cierpliwości ani zrozumienia dla ukraińskich racji. Błędnie zakładał też, że zdoła oderwać Rosję od sojuszu z Chinami. Tymczasem oba kraje mocno zacieśniły współpracę gospodarczą, a Chiny dostarczają dziś Rosjanom maszyny i technologie, na które monopol miał kiedyś Zachód. Moskwa i Pekin pamiętają czasy, gdy były podzielone. Putin i Xi są zgodni, że rozpad Związku Radzieckiego nie był koniecznością, tylko skutkiem błędów Kremla. Rosję i Chiny dzieli przepaść kulturalna, bo Rosjanie nie chcą być Chińczykami, ale to akurat ułatwia relacje. Współpraca z Europą grozi w oczach Putina zarażeniem rosyjskiego społeczeństwa „demokratycznym wirusem”, a w kontaktach z Chinami takiego ryzyka nie ma. Rosja może jednak wpaść w nieodwracalną zależność od Chin. Decydując się na inwazję na Ukrainę, Putin chciał przede wszystkim zasilić ludność Rosji dziesiątkami milionów Słowian, co było dla niego ważniejsze od zdobyczy terytorialnych. W tle są obawy przed chińską dominacją."""

In [ ]:
summary_generator_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert assistant for generating high-quality summaries of provided text.

Rules:
Return ONLY a valid summary string in Polish, do NOT include markdown, comments, or any text beside the summary.
Resolve ambiguities by clearly attributing statements, claims, or actions to specific people or entities, indicating who thinks, says, or does what, rather than leaving information vague or general.
The summary must be factual, neutral, and concise.
Your summary should consist of 3 to 4 sentences only.
Do NOT add interpretations, opinions, speculation, or external knowledge.
Preserve causal relationships, actors, and concrete actions.
Remove rhetoric, narrative framing, and redundancy.
Prefer explicit facts over vague generalizations.
Preserve exact numbers, dates, and named entities when present.
Focus on what happened and who did what; include claims only if they are themselves materially relevant events."""),
    ("user", """Generate summary of the following text:\n{text}""")
])

llm_chain = summary_generator_prompt | llm_text
summary = llm_chain.invoke({"text": text}).content
print(summary)

In [ ]:
allowed_nodes = [
    "Person",
    "Country",
    "Organization",
    "Alliance",
    "Conflict",
    "Event",
    "Ideology",
    "Technology",
    "EconomicSector",
    "Culture",
    "HistoricalEvent",
    "Policy"
]
allowed_relationships = [
    "PERCEIVES_AS",
    "ASSUMES_POSSIBILITY_OF",
    "STRENGTHENS_COOPERATION",
    "SUPPLIES_TECHNOLOGY",
    "DRAWS_HISTORICAL_LESSONS",
    "DENIES_NECESSITY_OF",
    "DIVIDES",
    "FEARS_DEMOCRATIZATION",
    "BECOMES_DEPENDENT_ON",
    "AIMS_FOR_DEMOGRAPHIC_REINFORCEMENT",
    "OPINION",
    "DECIDED_TO"
]
node_properties=True
relationship_properties=True
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships,
    node_properties=node_properties,
    relationship_properties=relationship_properties,
    strict_mode=False,
    additional_instructions="""Extract ONLY most important nodes (Events, Entities, Concepts etc.) and 10-12 most important factual relationships between them.

Critical rules (strict):
- Write extracted nodes and relationships in Polish.
- DO NOT extract rhetorical, or trivial relationships.
- Relationships must describe objective actions, dependencies, or states with any significance.
- Each relationship must have a clear standalone meaning outside the sentence context (a single “X -> Y” must convey real information on its own).

Importance filter:
- If a relationship does not change the understanding of the situation, omit it.
- Remove duplicates.""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(summary)])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")

In [ ]:
allowed_nodes = [
    "Osoba",
    "Państwo",
    "Organizacja",
    "Sojusz",
    "Konflikt",
    "Lokalizacja",
    "Pojęcie",
    "Wydarzenie",
    "Ideologia",
    "Technologia",
    "SektorGospodarki",
    "Kultura",
    "WydarzenieHistoryczne",
    "Polityka",

    "Prawo",
    "Instytucja",
    "Program",
    "Doktryna",
    "Zasób",
    "Proces",
    "Strategia",
    "Umowa",
    "Rynek",
    "ZjawiskoSpołeczne"
]
allowed_relationships = [
    "ZAATAKOWAŁ",
    "OKUPUJE",
    "BRONI_SIĘ_PRZED",
    "DOSTARCZA_BROŃ",
    "GROZI_UŻYCIEM_SIŁY",
    "ZAKOŃCZYŁ_KONFLIKT",

    "NAKŁADA_SANKCJE_NA",
    "ZAWIERA_SOJUSZ_Z",
    "ZERWAŁ_RELACJE_Z",
    "NEGOCJUJE_Z",
    "WSPIERA_POLITYCZNIE",
    "POTĘPIA_DZIAŁANIA",
    "UZNAJE_NIEPODLEGŁOŚĆ",

    "JEST_PRZYWÓDCĄ",
    "KONTROLUJE",
    "JEST_CZĘŚCIĄ",
    "REPREZENTUJE",
    "ZASTĘPUJE",

    "POWODUJE",
    "ZAPOBIEGA",
    "PRZYSPIESZA",
    "DESTABILIZUJE",
    "UMOŻLIWIA",
    "BLOKUJE",
    "ZAGRAŻA",
    "WYZNACZA_KIERUNEK",

    "FINANSUJE",
    "UZALEŻNIA_EKONOMICZNIE",
    "EKSPORTUJE_DO",
    "PRODUKUJE",
    "WYKORZYSTUJE_TECHNOLOGIĘ",

    "OBAWIA_SIĘ",
    "PLANUJE",
    "SPRZECIWIA_SIĘ",
    "KRYTYKUJE"
]
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships,
    node_properties=True,
    relationship_properties=True,
    strict_mode=False,
    additional_instructions="""Rules:
- Output language: Polish.
- Extract 20-25 the most important nodes (entities, concepts, events) and 10-15 most important relationships between them from provided summary of a text (maybe less if not possible).
- Each triplet (Node, Relation, Node) must be a complete, standalone factual statement.
- Relationship labels must be specific and informative, not vague or generic. Avoid single-word relationships like "UWAŻAŁ", "JEST" etc. where possible.
- Do not create meta-nodes (e.g., Autor, Tekst, Artykuł). If the author refers to themselves in the first person ('I believe...', 'My analysis...'), extract the core fact itself.""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(summary)])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")

In [ ]:
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    node_properties=True,
    relationship_properties=True,
    strict_mode=False,
    additional_instructions="""Rules:
- Output language: Polish.
- Extract 20-25 most important nodes (entities, concepts, events) and 10-15 most important relationships between them from provided text (fewer if the text does not support more without loss of quality).
- Each triplet (Node, Relation, Node) must be a complete, standalone factual statement.
- Relationship labels must be specific and informative, not vague or generic. Avoid single-word relationships like "UWAŻAŁ", "JEST" etc. where possible.
- Do not create meta-nodes (e.g., Autor, Tekst, Artykuł). First-person statements should be reduced to the **underlying objective fact** connecting real entities.""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(summary)])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")